# Data Basics Notebook
This notebook provides an introduction to working with both the Transfermarkt API
and UnderstatAPIs.

In [1]:
import duckdb
from understatapi import UnderstatClient
from matplotlib import pyplot as plt
import pandas as pd

## Transfermarkt API

In [3]:
# Example: Liverpool players from 2025
q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
WHERE current_club_domestic_competition_id = 'GB1' AND last_season = 2025
AND P.current_club_id = 31
"""

duckdb.sql(q).show() # Run the query


┌───────────┬────────────┬──────────────┬─────────────────────┬─────────────┬─────────────────┬─────────────────────┬───────────────────────┬────────────────────┬────────────────────────┬─────────────────────┬────────────────────┬────────────┬─────────┬──────────────┬──────────────────────────┬───────────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────────┬─────────────────────┬─────────────────────────────┐
│ player_id │ first_name │  last_name   │        name         │ last_season │ current_club_id │     player_code     │   country_of_birth    │   city_of_birth    │ country_of_citizenship │    date_of_birth    │    sub_position    │  position  │  foot   │ height_in_cm │ contract_expiration_date │    agent_name     │                                     image_url                                      │      

In [4]:
# Example: liverpool current player valuations over time

q = """
SELECT P.player_id, P.name, V.date, V.market_value_in_eur
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
WHERE current_club_domestic_competition_id = 'GB1' AND last_season = 2025
AND P.current_club_id = 31
ORDER BY P.player_id, V.date
"""

duckdb.sql(q).show() # Run the query

┌───────────┬─────────────────┬────────────┬─────────────────────┐
│ player_id │      name       │    date    │ market_value_in_eur │
│   int64   │     varchar     │    date    │        int64        │
├───────────┼─────────────────┼────────────┼─────────────────────┤
│    105470 │ Alisson         │ 2012-05-12 │              150000 │
│    105470 │ Alisson         │ 2012-08-02 │              150000 │
│    105470 │ Alisson         │ 2014-07-08 │              300000 │
│    105470 │ Alisson         │ 2015-01-02 │             1000000 │
│    105470 │ Alisson         │ 2015-11-09 │             3500000 │
│    105470 │ Alisson         │ 2017-02-01 │             7000000 │
│    105470 │ Alisson         │ 2017-08-06 │             7000000 │
│    105470 │ Alisson         │ 2018-07-06 │            60000000 │
│    105470 │ Alisson         │ 2019-10-12 │            90000000 │
│    105470 │ Alisson         │ 2020-08-04 │            72000000 │
│       ·   │    ·            │     ·      │                · 

In [5]:
# Example: Basic Performance Stats

q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/appearances.csv.gz') A
LIMIT 10
"""

duckdb.sql(q).show()

┌────────────────┬─────────┬───────────┬────────────────┬────────────────────────┬────────────┬──────────────────┬────────────────┬──────────────┬───────────┬───────┬─────────┬────────────────┐
│ appearance_id  │ game_id │ player_id │ player_club_id │ player_current_club_id │    date    │   player_name    │ competition_id │ yellow_cards │ red_cards │ goals │ assists │ minutes_played │
│    varchar     │  int64  │   int64   │     int64      │         int64          │    date    │     varchar      │    varchar     │    int64     │   int64   │ int64 │  int64  │     int64      │
├────────────────┼─────────┼───────────┼────────────────┼────────────────────────┼────────────┼──────────────────┼────────────────┼──────────────┼───────────┼───────┼─────────┼────────────────┤
│ 2231978_38004  │ 2231978 │     38004 │            853 │                    235 │ 2012-07-03 │ Aurélien Joachim │ CLQ            │            0 │         0 │     2 │       0 │             90 │
│ 2233748_79232  │ 2233748 │  

## Understat API

In [2]:
# Create connection
understat = UnderstatClient()

In [3]:
# Get all player data for a league/season
epl_players = understat.league(league="EPL").get_player_data(season="2025")

# Make into a dataframe
player_df = pd.DataFrame(epl_players)

player_df.head()

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
0,8260,Erling Haaland,29,2439,22,23.13693391531706,7,4.766327390447259,102,21,1,0,F S,Manchester City,19,20.092258475720882,26.238575093448162,4.129314249381423
1,13222,Thiago,31,2662,19,21.101609505712986,1,3.155416488647461,71,19,6,0,F S,Brentford,13,15.773427560925484,19.4751720353961,3.4934082627296448
2,11363,Antoine Semenyo,29,2597,15,11.201724836602807,4,3.091043023392558,68,32,6,0,F M,"Bournemouth,Manchester City",14,9.67938713915646,16.244873739778996,4.950852746143937
3,8272,João Pedro,31,2345,14,14.15835139900446,5,3.9609613977372646,63,28,4,0,F M S,Chelsea,14,14.15835139900446,17.67835896089673,4.659978250041604
4,501,Danny Welbeck,30,1832,12,10.899193301796913,0,1.8765279427170753,43,18,5,0,F S,Brighton,11,8.6156866569072,12.241745796054602,3.454270198009908


In [4]:
player_df.sort_values(by="time", ascending=False).head(20)

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
218,9558,Hannibal Mejbri,23,996,1,0.4494898612610996,3,1.9867054745554924,14,14,7,0,M S,Burnley,1,0.4494898612610996,4.205114980228245,2.764857292175293
459,12910,Mads Hermansen,11,990,0,0,0,0,0,0,0,0,GK,West Ham,0,0,1.1277148462831974,1.1277148462831974
56,5304,Mikel Merino,21,977,4,4.551519326865673,3,2.2901399955153465,21,12,1,0,F M S,Arsenal,4,4.551519326865673,7.0440468192100525,3.1536439023911953
435,11808,Andrey Santos,23,977,0,1.7154290862381458,0,0.6522208489477634,8,7,4,0,M S,Chelsea,0,1.7154290862381458,5.09560789167881,3.811923023313284
229,10770,Soungoutou Magassa,20,972,1,0.6193040320649743,0,0.7463846588507295,7,5,4,0,M S,West Ham,1,0.6193040320649743,2.4097342267632484,1.522645689547062
466,13150,Jack Fletcher,3,97,0,0.037386249750852585,0,0.11487564817070961,1,2,0,0,S,Manchester United,0,0.037386249750852585,0.26276974007487297,0.14789408445358276
217,9501,Fabio Carvalho,6,97,1,0.6895682029426098,0,0,2,0,0,0,M S,Brentford,1,0.6895682029426098,0.6895682029426098,0
203,8252,Nicolás Domínguez,20,965,1,0.7576484093442559,0,0.7514081094413996,16,9,1,0,M S,Nottingham Forest,1,0.7576484093442559,3.407697480171919,2.2724769935011864
298,5355,Jacob Bruun Larsen,23,965,0,1.3876315392553806,1,1.2826955989003181,16,11,0,0,F M S,Burnley,0,1.3876315392553806,4.335738031193614,2.199708631262183
393,10120,Conor Bradley,15,962,0,0.21982916723936796,1,1.1937107481062412,6,4,5,0,D S,Liverpool,0,0.21982916723936796,3.412436105310917,2.4054819140583277


In [38]:
player_df[player_df["player_name"] == "Antoine Semenyo"]

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
2,11363,Antoine Semenyo,29,2597,15,11.201724836602807,4,3.091043023392558,68,32,6,0,F M,"Bournemouth,Manchester City",14,9.67938713915646,16.244873739778996,4.950852746143937


In [5]:
# Look at all positions

player_df["position"].unique()

array(['F S', 'F M', 'F M S', 'M S', 'M', 'D M', 'D M S', 'D S', 'D',
       'D F M S', 'F', 'S', 'GK', 'GK S'], dtype=object)

In [29]:
# Get all match stats for a particular player - Erling Haaland
player_matches = understat.player(player='8260').get_match_data()

# Convert to dataframe
player_matches_df = pd.DataFrame(player_matches)

# Sort by date
player_matches_df = player_matches_df.sort_values(by="date")

player_matches_df.head()

,goals,shots,xG,time,position,h_team,a_team,h_goals,a_goals,date,id,season,roster_id,xA,assists,key_passes,npg,npxG,xGChain,xGBuildup
192,3,3,1.3227900266647339,33,Sub,Augsburg,Borussia Dortmund,3,5,2020-01-18,12562,2019,389276,0,0,0,3,1.3227900266647339,1.3459999561309814,0.023214099928736687
191,2,3,1.3333300352096558,23,Sub,Borussia Dortmund,FC Cologne,5,1,2020-01-24,12566,2019,389900,0,0,0,2,1.3333300352096558,1.3333300352096558,0
190,2,2,0.7829639911651611,77,FW,Borussia Dortmund,Union Berlin,5,0,2020-02-01,12574,2019,391067,0.06835760176181793,0,1,2,0.7829639911651611,0.7862169742584229,0.5620520114898682
189,0,2,0.10662200301885605,90,FW,Bayer Leverkusen,Borussia Dortmund,4,3,2020-02-08,12584,2019,392772,0,0,0,0,0.10662200301885605,0.422342985868454,0.3633730113506317
188,1,2,0.6940990090370178,80,FW,Borussia Dortmund,Eintracht Frankfurt,4,0,2020-02-14,12592,2019,393399,0.04634920135140419,0,1,1,0.6940990090370178,0.7866899967193604,0.046242501586675644


In [ ]:
team_season_leagues = pd.read_csv("../data/team_season_leagues.csv")

pd.merge(player_matches_df, team_season_leagues)

In [39]:
understat.player(player='11363').get_season_data()['season']

[{'position': 'MR',
  'games': '20',
  'goals': '10',
  'shots': '49',
  'time': '1800',
  'xG': '7.9325351398438215',
  'assists': '3',
  'xA': '2.4660277236253023',
  'key_passes': '25',
  'season': '2025',
  'team': 'Bournemouth',
  'yellow': '6',
  'red': '0',
  'npg': '9',
  'npxG': '6.410197442397475',
  'xGChain': '10.571022145450115',
  'xGBuildup': '3.088812258094549'},
 {'position': 'MR',
  'games': '9',
  'goals': '5',
  'shots': '19',
  'time': '797',
  'xG': '3.2691896967589855',
  'assists': '1',
  'xA': '0.6250152997672558',
  'key_passes': '7',
  'season': '2025',
  'team': 'Manchester City',
  'yellow': '0',
  'red': '0',
  'npg': '5',
  'npxG': '3.2691896967589855',
  'xGChain': '5.67385159432888',
  'xGBuildup': '1.862040488049388'},
 {'position': 'FW',
  'games': '37',
  'goals': '11',
  'shots': '126',
  'time': '3219',
  'xG': '14.549346312880516',
  'assists': '5',
  'xA': '6.608794005587697',
  'key_passes': '46',
  'season': '2024',
  'team': 'Bournemouth',
  '

In [12]:
# Get all player names and ids and save to csv for later use

leagues = ["EPL", "La_Liga", "Bundesliga", "Serie_A", "Ligue_1"]
seasons = list(range(2015, 2026))

id_to_name = {}
for league in leagues:
    for season in seasons:
        players = understat.league(league=league).get_player_data(season=str(season))
        for p in players:
            id_to_name[p["id"]] = p["player_name"]
            
id_names = pd.DataFrame({"player_id":id_to_name.keys(), "name":id_to_name.values()})
id_names.to_csv("../data/id_names.csv")

In [28]:
# Get mapping from team/season to league
team_season_to_league = {}
for league in leagues:
    for season in seasons:
        teams = understat.league(league=league).get_team_data(season=str(season))
        for team in teams:
            team_season_to_league[(teams[team]['title'], season)] = league

# Save as df
team_season_leagues = pd.DataFrame({"team":[k[0] for k in team_season_to_league.keys()],
                                   "season":[k[1] for k in team_season_to_league.keys()],
                                   "league":[v for v in team_season_to_league.values()]})

team_season_leagues.to_csv("../data/team_season_leagues.csv")

In [42]:
understat.league(league="EPL").get_match_data(season="2025")

[{'id': '28778',
  'isResult': True,
  'h': {'id': '87', 'title': 'Liverpool', 'short_title': 'LIV'},
  'a': {'id': '73', 'title': 'Bournemouth', 'short_title': 'BOU'},
  'goals': {'h': '4', 'a': '2'},
  'xG': {'h': '2.33007', 'a': '1.57303'},
  'datetime': '2025-08-15 19:00:00',
  'forecast': {'w': '0.5498', 'd': '0.2276', 'l': '0.2226'}},
 {'id': '28779',
  'isResult': True,
  'h': {'id': '71', 'title': 'Aston Villa', 'short_title': 'AVL'},
  'a': {'id': '86', 'title': 'Newcastle United', 'short_title': 'NEW'},
  'goals': {'h': '0', 'a': '0'},
  'xG': {'h': '0.318601', 'a': '1.40098'},
  'datetime': '2025-08-16 11:30:00',
  'forecast': {'w': '0.0629', 'd': '0.2371', 'l': '0.7'}},
 {'id': '28780',
  'isResult': True,
  'h': {'id': '220', 'title': 'Brighton', 'short_title': 'BRI'},
  'a': {'id': '228', 'title': 'Fulham', 'short_title': 'FLH'},
  'goals': {'h': '1', 'a': '1'},
  'xG': {'h': '1.44008', 'a': '0.902883'},
  'datetime': '2025-08-16 14:00:00',
  'forecast': {'w': '0.5158', '

In [57]:
from collections import defaultdict

player_info_map = defaultdict(list)

# Get all matches in this league/season
matches = understat.league(league=league).get_match_data(season=str(season))
        
for match in matches:
    match_id = match['id']
    # Get player stats for this specific match
    roster = understat.match(match=match_id).get_roster_data()
            
    for _, player_info in roster['h'].items():  # home team
        player_info_map[player_info['player_id']].append(player_info)
                
    for _, player_info in roster['a'].items():  # away team
        player_info_map[player_info['player_id']].append(player_info)

In [2]:
understat.match(match='30520').get_roster_data()

NameError: name 'understat' is not defined

In [7]:
understat.league(league="EPL").get_match_data(season="2025")[0]['datetime'][:10]

'2025-08-15'

In [19]:
understat.match(match='30467').get_roster_data()

InvalidMatch: 30467 is not a valid match

In [60]:
pd.DataFrame(player_info_map['560'])

,id,goals,own_goals,shots,xG,time,player_id,team_id,position,player,...,yellow_card,red_card,roster_in,roster_out,key_passes,assists,xA,xGChain,xGBuildup,positionOrder
0,620382,0,0,0,0,90,560,89,GK,Sergio Romero,...,0,0,0,0,0,0,0,0,0,1
1,57370,0,0,0,0,90,560,89,GK,Sergio Romero,...,0,0,0,0,0,0,0,0.0589383989572525,0.0589383989572525,1
2,62806,0,0,0,0,90,560,89,GK,Sergio Romero,...,0,0,0,0,0,0,0,0.014860300347208977,0.014860300347208977,1
3,63779,0,0,0,0,90,560,89,GK,Sergio Romero,...,0,0,0,0,0,0,0,0.5704770088195801,0.5704770088195801,1
